# Nettoyage de l'export journalier
**Responsable : Ihman** · export `Pharma_Ventes_Daily.csv` · module `src/pharma/cleaning/daily.py`

**Pourquoi ce nettoyage ?** Les 4 exports ont été produits séparément et personne n'a vérifié qu'ils concordent.
L'export horaire est la **référence** (notebook 01) ; l'export journalier sert de **contrôle** : on vérifie qu'il est
propre et que chaque jour égale la somme de ses heures. Les ventes retenues sont les **jours complets recalculés
depuis l'horaire**. Le client renverra ce fichier chaque mois : tous ces contrôles tournent dans le code à chaque fois.

## 0 · Préparer le notebook

In [1]:
import sys
from pathlib import Path

# Le notebook est dans notebooks/ : on remonte à la racine du projet pour importer le code
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from pharma import config
from pharma.cleaning import checks, hourly, loading

G = config.GROUP_CODES          # les 8 groupes ATC, dans l'ordre des colonnes

## 1 · Lire l'export brut, tel qu'il arrive du logiciel de caisse

In [2]:
brut = pd.read_csv(config.RAW_DIR / config.EXPORTS["daily"])
print(brut.shape)          # (nombre de lignes, nombre de colonnes)
brut.head()

(2106, 13)


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,Hour,Weekday Name
0,1/2/2014,0.0,3.67,3.4,32.40,7.0,0,0.0,2.0,2014,1,248,Thursday
1,1/3/2014,8.0,4.00,4.4,50.60,16.0,0,20.0,4.0,2014,1,276,Friday
2,1/4/2014,2.0,1.00,6.5,61.85,10.0,0,9.0,1.0,2014,1,276,Saturday
3,1/5/2014,4.0,3.00,7.0,41.10,8.0,0,3.0,0.0,2014,1,276,Sunday
4,1/6/2014,5.0,1.00,4.5,21.70,16.0,2,6.0,2.0,2014,1,276,Monday


In [3]:
brut.dtypes          # datum est du texte au format mois/jour/année (1/2/2014 = 2 janvier) : il faut le convertir

datum               str
M01AB           float64
M01AE           float64
N02BA           float64
N02BE           float64
N05B            float64
N05C              int64
R03             float64
R06             float64
Year              int64
Month             int64
Hour              int64
Weekday Name        str
dtype: object

**Lecture propre** : `loading.load_export` convertit les dates et garde les 8 groupes ;
les colonnes ajoutées par le logiciel (`Year`, `Month`, `Hour`, `Weekday Name`) sont lues à part, pour être contrôlées.

In [4]:
export = loading.load_export("daily")
derived = loading.load_derived_columns("daily")
print(f"{len(export)} jours, du {export.index.min():%d/%m/%Y} au {export.index.max():%d/%m/%Y}")
export.head()

2106 jours, du 02/01/2014 au 08/10/2019


,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06
date,,,,,,,,
2014-01-02,0.0,3.67,3.4,32.40,7.0,0,0.0,2.0
2014-01-03,8.0,4.00,4.4,50.60,16.0,0,20.0,4.0
2014-01-04,2.0,1.00,6.5,61.85,10.0,0,9.0,1.0
2014-01-05,4.0,3.00,7.0,41.10,8.0,0,3.0,0.0
2014-01-06,5.0,1.00,4.5,21.70,16.0,2,6.0,2.0


## 2 · Les contrôles communs (1 à 4 et 6)
Périmètre, doublons, valeurs manquantes, valeurs négatives, colonnes du logiciel cohérentes avec la date.

In [5]:
journal = []                                                  # chaque contrôle ajoute une ligne
checks.scope(journal, "daily", export)                                  # 1 · périmètre
export = checks.duplicates_and_negatives(journal, "daily", export)     # 2, 3, 4 · doublons, manquants, négatifs
checks.derived_columns(journal, "daily", derived)                       # 6 · Year, Month, Weekday Name
pd.DataFrame(journal)

,Contrôle,Export,Statut,Détail,Décision
0,Périmètre,daily,OK,"2106 lignes, du 02/01/2014 au 08/10/2019",Information
1,Doublons de date,daily,OK,0 date(s) en double,Aucune action
2,Valeurs manquantes,daily,OK,0 valeur(s) manquante(s),Aucune action
3,Valeurs négatives,daily,OK,0 valeur(s) négative(s),Aucune action
4,Colonnes dérivées cohérentes avec la date,daily,OK,"Year : 0 incohérence(s), Month : 0 incohérence...",Colonnes ignorées : toujours recalculées depui...


## 3 · Continuité : manque-t-il des jours ? (contrôle 5)

In [6]:
tous_les_jours = pd.date_range(export.index.min(), export.index.max(), freq="D")
manquants = tous_les_jours.difference(export.index)
print(f"{len(manquants)} jour(s) manquant(s) entre le premier et le dernier jour")

0 jour(s) manquant(s) entre le premier et le dernier jour


## 4 · La colonne `Hour` : un artefact qui révèle les jours incomplets
Dans le journalier, `Hour` n'est pas une heure : c'est la **somme des heures présentes** ce jour-là dans l'horaire.
Un jour complet a les 24 heures : 0 + 1 + … + 23 = **276**. Une valeur plus petite signale des heures manquantes.

In [7]:
print("Jour complet :", sum(range(24)))
derived["Hour"].value_counts()          # combien de jours pour chaque valeur de Hour

Jour complet : 276


Hour
276    2104
248       1
190       1
Name: count, dtype: int64

In [8]:
# Les jours dont Hour n'est pas 276, et ce que vaut la somme des heures attendues
incomplets = derived[derived["Hour"] != 276][["Hour", "Weekday Name"]]
incomplets["commence à 8h (8+...+23)"] = sum(range(8, 24))
incomplets["s'arrête à 19h (0+...+19)"] = sum(range(20))
incomplets

,Hour,Weekday Name,commence à 8h (8+...+23),s'arrête à 19h (0+...+19)
date,,,,
2014-01-02,248,Thursday,248,190
2019-10-08,190,Tuesday,248,190


Le premier jour (248 = 8 + … + 23) commence à 8 h, le dernier (190 = 0 + … + 19) s'arrête à 19 h.
On ne garde pas la colonne `Hour` pour l'analyse, mais elle confirme ce que montre l'horaire.

## 5 · Le journalier concorde-t-il avec l'horaire ? (contrôle 9)

In [9]:
# La référence : l'export horaire nettoyé par Adelaida (hourly.py)
horaire = hourly.clean_hourly(loading.load_export("hourly"), loading.load_derived_columns("hourly"), [])
daily_all = horaire.daily_all           # ventes par jour recalculées depuis l'horaire (tous les jours)
complete_day = horaire.complete_day     # True si le jour est complet selon l'horaire
print(f"{len(daily_all)} jours recalculés, dont {int(complete_day.sum())} complets")

2106 jours recalculés, dont 2105 complets


In [10]:
ecarts = checks.compare(export, daily_all, "daily", "%d/%m/%Y")   # écart toléré : arrondis seulement
print(f"{len(ecarts)} case(s) en écart sur {len(export) * len(G)}")
ecarts

0 case(s) en écart sur 16848


,Export,Période,Groupe,Valeur export,Valeur recalculée (horaire),Écart,Écart %


## 6 · Jours incomplets : exclus des ventes propres (contrôle 7)

In [11]:
exclus = complete_day[~complete_day].index
print("Jours exclus (incomplets selon l'horaire) :", [f"{d:%d/%m/%Y}" for d in exclus])
ventes = daily_all.loc[complete_day[complete_day].index, G]
print(f"Ventes propres : {len(ventes)} jours complets, du {ventes.index.min():%d/%m/%Y} au {ventes.index.max():%d/%m/%Y}")

Jours exclus (incomplets selon l'horaire) : ['08/10/2019']
Ventes propres : 2105 jours complets, du 02/01/2014 au 07/10/2019


Le 02/01/2014 n'est **pas** exclu : il ne lui manque que les heures de nuit (0 h à 7 h), où il n'y a presque pas
de ventes (décision du notebook 01). Le 08/10/2019, arrêté à 19 h, l'est.

## 7 · Pics inhabituels (contrôle 8) : signalés, jamais supprimés

In [12]:
pics = checks.flag_outliers(ventes)
print(f"{len(pics)} pics signalés, par groupe :")
print(pics["Groupe"].value_counts().to_string())
pics.sort_values("Quantité", ascending=False).head(8)

58 pics signalés, par groupe :
Groupe
R03      44
N05B      6
N02BE     5
M01AE     2
R06       1


,Date,Groupe,Quantité,Médiane 29 j,z robuste,Décision
33,2017-01-13,N02BE,161.00,46.0,6.9,"Conservé (pas de preuve d'erreur), signalé"
43,2017-10-08,N02BE,77.10,39.2,6.1,"Conservé (pas de preuve d'erreur), signalé"
41,2017-10-01,N02BE,72.10,38.0,6.1,"Conservé (pas de preuve d'erreur), signalé"
32,2016-10-11,N05B,54.83,5.0,11.2,"Conservé (pas de preuve d'erreur), signalé"
48,2018-05-27,N02BE,49.26,19.0,6.0,"Conservé (pas de preuve d'erreur), signalé"
52,2019-02-02,R03,45.00,8.0,6.2,"Conservé (pas de preuve d'erreur), signalé"
37,2017-08-23,N02BE,42.00,18.4,6.1,"Conservé (pas de preuve d'erreur), signalé"
44,2018-01-17,R03,41.00,5.0,8.1,"Conservé (pas de preuve d'erreur), signalé"


## Étape finale · Le module `daily.py`, tel que `main.py` l'utilise
Mêmes contrôles, en une seule fonction (contrat : `clean_daily(export, derived, daily_all, complete_day, rows) -> (daily, ecarts)`).
Le journal produit remplira la feuille **État des données** du classeur.

In [13]:
from pharma.cleaning.daily import clean_daily

journal_final = []
daily, ecarts_final = clean_daily(loading.load_export("daily"), derived, daily_all, complete_day, journal_final)
print(f"{len(daily)} jours complets ; {len(ecarts_final)} écart avec l'horaire")
pd.DataFrame(journal_final)

2105 jours complets ; 0 écart avec l'horaire


,Contrôle,Export,Statut,Détail,Décision
0,Périmètre,daily,OK,"2106 lignes, du 02/01/2014 au 08/10/2019",Information
1,Doublons de date,daily,OK,0 date(s) en double,Aucune action
2,Valeurs manquantes,daily,OK,0 valeur(s) manquante(s),Aucune action
3,Valeurs négatives,daily,OK,0 valeur(s) négative(s),Aucune action
4,Jours manquants dans la suite,daily,OK,0 jour(s) manquant(s),Aucune action
5,Colonnes dérivées cohérentes avec la date,daily,OK,"Year : 0 incohérence(s), Month : 0 incohérence...",Colonnes ignorées : toujours recalculées depui...
6,Jours incomplets selon la colonne Hour (≠ 276),daily,ANOMALIE,"02/01/2014 (Hour = 248), 08/10/2019 (Hour = 190)",Information : l'horaire décide quels jours son...
7,Jours incomplets exclus,daily,ANOMALIE,08/10/2019,Exclus des statistiques journalières
8,Réconciliation avec l'horaire,daily,OK,0 période(s) en écart sur 2106 (0 cellule(s)),Concorde : export validé


## Ce que je retiens

**Constat 1 · Un fichier propre.** 2 106 jours, du 02/01/2014 au 08/10/2019. Aucun doublon, aucune case vide, aucune vente
négative, aucun jour manquant. `Year`, `Month` et `Weekday Name` concordent avec la date sur toutes les lignes.

**Constat 2 · Il concorde avec l'horaire.** Chaque jour égale la somme de ses heures (0 case en écart) : le journalier est
fiable, mais on ne l'utilise que comme contrôle.

**Constat 3 · La colonne `Hour` est un artefact utile.** Elle vaut 276 pour un jour complet ; 248 le premier jour (début à 8 h)
et 190 le dernier (arrêt à 19 h). Le dernier jour est exclu : **2 105 jours complets** retenus, jusqu'au 07/10/2019.

**Constat 4 · 58 pics signalés, conservés.** Surtout R03 (asthme), groupe aux ventes faibles où un seul gros achat suffit ;
on retrouve aussi N02BE à 161 le 13/01/2017 et N05B à 54,8 le 11/10/2016. Rien ne prouve une erreur de saisie : on ne corrige pas.

**Limite.** Ventes 7 jours sur 7, sans aucun jour fermé, même les jours fériés : peu typique d'une officine française.